In [ ]:
# rtmdet-tiny, coco, plantdoc, epoch 300, simam注意力机制
!python tools/train.py configs/rtmdet/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam.py
!featurize instance release $UUID

06/29 22:48:25 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.7.10 (default, Jun  4 2021, 14:48:32) [GCC 7.5.0]
    CUDA available: True
    numpy_random_seed: 833253592
    GPU 0: NVIDIA GeForce RTX 3090
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.2, V11.2.152
    GCC: gcc (Ubuntu 9.3.0-17ubuntu1~20.04) 9.3.0
    PyTorch: 1.10.0+cu113
    PyTorch compiling details: PyTorch built with:
  - GCC 7.3
  - C++ Version: 201402
  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.2.3 (Git Hash 7336ca9f055cf1bfa13efb658fe15dc9b41f0740)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.3
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,co

In [1]:
# test
!python tools/test.py configs/rtmdet/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam.py \
work_dirs/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam/best_coco_bbox_mAP_epoch_298.pth

06/30 08:44:07 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.7.10 (default, Jun  4 2021, 14:48:32) [GCC 7.5.0]
    CUDA available: True
    numpy_random_seed: 658943898
    GPU 0: NVIDIA GeForce RTX 3090
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.2, V11.2.152
    GCC: gcc (Ubuntu 9.3.0-17ubuntu1~20.04) 9.3.0
    PyTorch: 1.10.0+cu113
    PyTorch compiling details: PyTorch built with:
  - GCC 7.3
  - C++ Version: 201402
  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.2.3 (Git Hash 7336ca9f055cf1bfa13efb658fe15dc9b41f0740)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.3
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,co

In [4]:
# 计算参数量和FLOPS
!python tools/analysis_tools/get_flops.py configs/rtmdet/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam.py

06/30 17:39:11 - mmengine - WARNING - Some config files, such as configs/yolact and configs/detectors,may have compatibility issues with torch.jit when torch<1.12. If you want to calculate flops for these models, please make sure your pytorch version is >=1.12.
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::silu_ encountered 45 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::add encountered 9 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::hardsigmoid_ encountered 4 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::mul encountered 47 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::max_pool2d encountered 3 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::sigmoid encountered 34 time(s)
06/30 17:39:31 - mmengine - WARNING - Unsupported operator aten::sub encountered 6 t

In [2]:
# calculate Params and FLOPS
import torch
from mmdet.apis import init_detector
from fvcore.nn import FlopCountAnalysis

config_path = "configs/rtmdet/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam.py"
checkpoint_path = "work_dirs/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam/best_coco_bbox_mAP_epoch_298.pth"
input_shape = (3, 640, 640)
input_h, input_w = 640, 640
device="cuda:0"

model = init_detector(config_path, checkpoint_path, device=device)
model.eval()

# params
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)

#  FLOPs
dummy_img = torch.randn(1, 3, input_h, input_w).cuda()
dummy_metas = [{"img_shape": (input_h, input_w, 3), "scale_factor": 1.0}]
inputs = (dummy_img, dummy_metas)

flop_analyzer = FlopCountAnalysis(model, inputs)
flops = flop_analyzer.total()   

print("="*60)
print(f"Input shape: 3 × {input_h} × {input_w}")
print(f"Total FLOPs: {flops/1e9:.3f}G")
print(f"Total Params: {total/1e6:.3f}M, Trainable: {trainable/1e6:.3f}M")
print("="*60) 

Loads checkpoint by local backend from path: work_dirs/rtmdet_tiny_8xb32-300e_coco-plantdoc-simam/best_coco_bbox_mAP_epoch_298.pth


Unsupported operator aten::silu_ encountered 45 time(s)
Unsupported operator aten::add encountered 9 time(s)
Unsupported operator aten::hardsigmoid_ encountered 4 time(s)
Unsupported operator aten::mul encountered 47 time(s)
Unsupported operator aten::max_pool2d encountered 3 time(s)
Unsupported operator aten::sigmoid encountered 34 time(s)
Unsupported operator aten::sub encountered 6 time(s)
Unsupported operator aten::mean encountered 3 time(s)
Unsupported operator aten::pow encountered 3 time(s)
Unsupported operator aten::sum encountered 3 time(s)
Unsupported operator aten::div encountered 6 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
bbox_head.loss_bbox, bbox_head.loss_centerness, bbox_head.loss_cls, d

Input shape: 3 × 640 × 640
Total FLOPs: 8.048G
Total Params: 4.881M, Trainable: 4.881M
